In [ ]:
!git clone -q https://github.com/cad0m/jarida-rag.git

In [ ]:
!pip install -q -e jarida-rag

In [ ]:
# Generation only: retrieval is frozen in config.py by notebooks/retrieval.ipynb; run that notebook again if it changes
import gc
import json
import os
import sys

from kaggle_secrets import UserSecretsClient

DAGSHUB_USER = "cad0m"
secrets = UserSecretsClient()

os.environ["JARIDA_PDF_DIR"] = "/kaggle/input/datasets/ayouberraouy/jarida"
os.environ["JARIDA_WORK"] = "/kaggle/working"
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{DAGSHUB_USER}/jarida-rag.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = secrets.get_secret("DAGSHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["DATABASE_URL"] = secrets.get_secret("NEON_DATABASE_URL")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation; must be set before torch loads
# the editable install is only on the path after a restart, so add it for this session
sys.path.insert(0, "/kaggle/working/jarida-rag/src")

import mlflow
import torch
from jarida_rag import config, rag, store
from jarida_rag.evaluation import generation, judge, retrieval

mlflow.set_tracking_uri(config.MLFLOW_URI)
# DagsHub answers a bare 404 for unknown experiment names, so look it up with search instead
found = mlflow.search_experiments(filter_string="name = 'gazette-rag'")
experiment_id = found[0].experiment_id if found else mlflow.create_experiment("gazette-rag")
mlflow.set_experiment(experiment_id=experiment_id)
RETRIEVER = ("hybrid" if config.HYBRID else "dense") + "+rerank"


def show_gpus():
    for gpu in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(gpu)
        print(f"GPU {gpu}: {free / 1024**3:.1f} GB free of {total / 1024**3:.1f} GB")


show_gpus()

In [ ]:
pool = store.connect()
embedder = store.load_embedder()
reranker = store.load_reranker()
print(store.build_index(pool, embedder), "chunks")
lexical_index = store.load_lexical(pool) if config.HYBRID else None
show_gpus()

In [ ]:
# Postgres must give the same numbers as the in-memory row in notebooks/retrieval.ipynb; if not, stop here
_, check = retrieval.evaluate(lambda question: store.candidates(pool, embedder, question, lexical_index), reranker, "retrieval-postgres", RETRIEVER)
check

In [ ]:
tokenizer, model = rag.load_llm()
print(f"LLM size: {model.get_memory_footprint() / 1024**3:.1f} GB")
show_gpus()

In [ ]:
for question in ["Which decree regulates marine aquaculture farms?", "What is the legal age of marriage in Morocco?"]:
    reply, hits = rag.answer(question, pool, embedder, tokenizer, model, reranker, lexical_index)
    print(question)
    print(reply)
    for hit in hits:
        print("  -", rag.source(hit), hit["title"][:60])
    print()

In [ ]:
# ~17 min for all 86 questions: re-run this cell after each system prompt change, the judge isn't needed
answers = generation.answer_questions(pool, embedder, tokenizer, model, reranker, lexical_index)
json.dump(answers, open("/kaggle/working/answers.json", "w"), ensure_ascii=False)
table, metrics = generation.evaluate(answers, "generation", RETRIEVER)
print(metrics)
print(table.groupby("type")["facts_ok"].agg(["sum", "count"]))
table[(table["facts_ok"] == False) | (table["refused"] != table["should_refuse"])][["type", "question", "answer"]]

In [ ]:
# RAGAS phase: the answers are saved, so free both GPUs completely (both should show ~14.5 GB free)
del model, tokenizer, embedder, reranker
gc.collect()
torch.cuda.empty_cache()
show_gpus()

In [ ]:
# After a session restart, skip the cells above (except setup) and load the saved answers instead
answers = json.load(open("/kaggle/working/answers.json"))

In [ ]:
judge_llm = judge.load_judge()
show_gpus()  # expected: the judge on GPU 0
# look at the raw output first: a thinking block or extra text here means RAGAS will fail to parse it
print(judge_llm.invoke('Reply with exactly this JSON and nothing else: {"ok": true}').content)

In [ ]:
# 2 questions first: if the scores are NaN, read the log above it before running the full evaluation
smoke = generation.ragas_eval([a for a in answers if a["type"] == "attribute"][:2], judge_llm, "ragas-smoke", RETRIEVER)
smoke

In [ ]:
# ~25 min, once the prompt is final: faithfulness of the 15 single-fact answers
scores = generation.ragas_eval(answers, judge_llm, "ragas", RETRIEVER)
scores["faithfulness"].mean()

In [ ]:
# Only after pushing new code: pull it and reload the modules without restarting
!git -C /kaggle/working/jarida-rag pull

import importlib
from jarida_rag import chunk, lexical, pdf

for module in [config, pdf, chunk, lexical, store, rag, retrieval, generation, judge]:
    importlib.reload(module)

# objects built before the reload (judge_llm, pool, ...) still run the old code: re-run the cells that create them